# Matemática dos HOS — Demonstração Computacional

**Caderno-companheiro de** [`HOS_math_tutorial.md`](HOS_math_tutorial.md) **e de** [`philosophical_essay_HOS.md`](philosophical_essay_HOS.md).

Este caderno realiza, em sinais sintéticos, o argumento central do tutorial: que a Densidade Espectral de Potência (PSD), por descartar fase, é incapaz de distinguir processos que diferem apenas na estrutura de acoplamento de fase entre suas componentes harmônicas — enquanto o biespectro, ao reter essa informação via produto triádico complexo, o faz de modo categórico.

A organização segue o tutorial:

1. **Sinais sintéticos com acoplamento de fase controlado** — duas populações de realizações: uma com $\varphi(f_1+f_2) = \varphi(f_1) + \varphi(f_2)$ (acoplada); outra com $\varphi(f_1+f_2)$ independente (não acoplada).
2. **PSDs lado a lado** — operacionalização de $\mathbb{E}[X(f_k)X^*(f_k)]$ por *segment averaging*, mostrando que as duas populações são indistinguíveis sob esse estimador (Sinha, 2007, Eq. 1).
3. **Biespectro** — implementação direta de $\mathbb{E}[X(f_l)X(f_m)X^*(f_l+f_m)]$ (Sinha, 2007, Eq. 2) e visualização das duas populações.
4. **Demonstração tipo-Sinha** — sinais sintéticos com padrões de acoplamento $B_{22}$ (tipo-trinca) e $B_{13}$ (tipo-desalinhamento), reproduzindo a geometria diagnóstica de Sinha (2007, Figs. 5–6) em sinais controlados.

Nenhum sinal de rotor real é usado neste caderno; o objetivo é didático e isola o aparato de HOS de toda a complexidade rotodinâmica.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=2026)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True

## 1. Sinais sintéticos com acoplamento de fase controlado

Geramos $K$ realizações independentes de duas populações de sinais. Cada realização tem três componentes harmônicas em $f_1$, $f_2$ e $f_3 = f_1 + f_2$, todas com a mesma amplitude e ruído gaussiano aditivo. As fases $\varphi_1$ e $\varphi_2$ são sorteadas uniformemente em $[0, 2\pi)$ a cada realização. A diferença é a regra para $\varphi_3$:

- **Acoplado:** $\varphi_3 = \varphi_1 + \varphi_2$ (fase do terceiro tom é deterministicamente travada — caso canônico de acoplamento quadrático de fase, tal como ocorreria se o terceiro tom fosse gerado por $\cos(2\pi f_1 t + \varphi_1) \cdot \cos(2\pi f_2 t + \varphi_2)$ sob uma não-linearidade multiplicativa).
- **Não acoplado:** $\varphi_3$ sorteado independentemente de $\varphi_1, \varphi_2$ (três processos não relacionados que por acaso oscilam em frequências aritmeticamente compatíveis).

Os parâmetros mantêm a relação de amostragem confortável: $f_s = 1000$ Hz, $N = 1024$ amostras por realização, $K = 400$ realizações. A resolução em frequência é $\Delta f = f_s/N \approx 0{,}98$ Hz — análoga em ordem de grandeza aos 1{,}25 Hz reportados em Sinha (2007, p. 329).

In [ ]:
fs = 1000.0
N = 1024
K = 400
f1, f2 = 50.0, 120.0
f3 = f1 + f2
amplitude = 1.0
noise_std = 0.3

t = np.arange(N) / fs


def make_ensemble(coupled: bool, K: int, rng: np.random.Generator) -> np.ndarray:
    ensemble = np.empty((K, N))
    for k in range(K):
        phi1 = rng.uniform(0.0, 2.0 * np.pi)
        phi2 = rng.uniform(0.0, 2.0 * np.pi)
        phi3 = phi1 + phi2 if coupled else rng.uniform(0.0, 2.0 * np.pi)
        s = (
            amplitude * np.cos(2.0 * np.pi * f1 * t + phi1)
            + amplitude * np.cos(2.0 * np.pi * f2 * t + phi2)
            + amplitude * np.cos(2.0 * np.pi * f3 * t + phi3)
            + noise_std * rng.standard_normal(N)
        )
        ensemble[k] = s
    return ensemble


coupled = make_ensemble(True, K, rng)
uncoupled = make_ensemble(False, K, rng)

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 5))
axes[0].plot(t[:256], coupled[0, :256], label='realização 0')
axes[0].plot(t[:256], coupled[1, :256], alpha=0.6, label='realização 1')
axes[0].set_title('Ensemble acoplado — primeiras 256 amostras de duas realizações')
axes[0].set_ylabel('amplitude')
axes[0].legend(loc='upper right')
axes[1].plot(t[:256], uncoupled[0, :256], label='realização 0')
axes[1].plot(t[:256], uncoupled[1, :256], alpha=0.6, label='realização 1')
axes[1].set_title('Ensemble não acoplado — primeiras 256 amostras de duas realizações')
axes[1].set_xlabel('tempo (s)')
axes[1].set_ylabel('amplitude')
axes[1].legend(loc='upper right')
plt.tight_layout()
plt.show()

Visualmente, ambas as populações parecem ruído harmônico de aparência aleatória. A diferença estrutural — o travamento da fase do terceiro tom — não se manifesta na forma de onda de uma realização isolada.

## 2. PSDs lado a lado — a cegueira da segunda ordem

Estimamos $\hat S_{xx}(f_k) = \frac{1}{K}\sum_{k=1}^{K} |X^{(k)}(f_k)|^2$, com $X^{(k)}$ a DFT da $k$-ésima realização (cada realização funcionando como um "segmento" da promediação descrita em Sinha, 2007, p. 329; é o promedio sobre realizações, não sobre janelas deslizantes — funcionalmente equivalente sob estacionariedade).

In [ ]:
def psd_estimate(ensemble: np.ndarray, fs: float) -> tuple[np.ndarray, np.ndarray]:
    X = np.fft.rfft(ensemble, axis=1)
    psd = np.mean(np.abs(X) ** 2, axis=0) / (ensemble.shape[1] * fs)
    freqs = np.fft.rfftfreq(ensemble.shape[1], d=1.0 / fs)
    return freqs, psd


freqs_psd, psd_coupled = psd_estimate(coupled, fs)
_, psd_uncoupled = psd_estimate(uncoupled, fs)

fig, ax = plt.subplots()
ax.semilogy(freqs_psd, psd_coupled, label='acoplado')
ax.semilogy(freqs_psd, psd_uncoupled, label='não acoplado', linestyle='--')
ax.set_xlim(0, 300)
ax.set_xlabel('frequência (Hz)')
ax.set_ylabel(r'$\hat S_{xx}(f)$  (escala log)')
ax.set_title('PSD estimada — duas populações são visualmente idênticas')
for f in (f1, f2, f3):
    ax.axvline(f, color='grey', linewidth=0.5, alpha=0.6)
ax.legend()
plt.tight_layout()
plt.show()

print(f'Máx |PSD_acoplado − PSD_não-acoplado| / PSD_max ≈ '
      f'{np.max(np.abs(psd_coupled - psd_uncoupled)) / psd_coupled.max():.3e}')

As duas PSDs são, dentro da variância amostral, **indistinguíveis**: os três picos aparecem em $f_1$, $f_2$ e $f_3$ com a mesma altura nas duas populações. Esta é a verificação operacional do alerta de Oppenheim, Willsky e Nawab (1997, p. 424): manter as magnitudes e variar as fases produz "very different-looking signals, even if the magnitude function remains unchanged" — mas a *magnitude espectral promediada* é, ela própria, exatamente igual.

## 3. Biespectro — o terceiro grau estatístico revela o acoplamento

Implementamos diretamente $\hat B_{xxx}(f_l, f_m) = \frac{1}{K}\sum_k X^{(k)}(f_l)\,X^{(k)}(f_m)\,X^{(k)*}(f_l + f_m)$, restrito ao triângulo principal $l + m \leq N/2$ (Sinha, 2007, Eq. 2). A grandeza retornada é complexa; mostramos seu módulo.

In [ ]:
def bispectrum_estimate(ensemble: np.ndarray, fs: float, f_max: float) -> tuple[np.ndarray, np.ndarray]:
    N = ensemble.shape[1]
    X = np.fft.fft(ensemble, axis=1)
    full_freqs = np.fft.fftfreq(N, d=1.0 / fs)
    half = N // 2
    keep = np.searchsorted(full_freqs[:half], f_max) + 1
    B = np.zeros((keep, keep), dtype=complex)
    for l in range(keep):
        for m in range(keep):
            lm = l + m
            if lm < half:
                B[l, m] = np.mean(X[:, l] * X[:, m] * np.conj(X[:, lm]))
    freqs = full_freqs[:keep]
    return freqs, np.abs(B)


f_max_plot = 250.0
freqs_bs, B_coupled = bispectrum_estimate(coupled, fs, f_max_plot)
_, B_uncoupled = bispectrum_estimate(uncoupled, fs, f_max_plot)

B_max = max(B_coupled.max(), B_uncoupled.max())
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
for ax, B, title in zip(axes, (B_coupled, B_uncoupled), ('acoplado', 'não acoplado')):
    im = ax.pcolormesh(freqs_bs, freqs_bs, B / B_max, cmap='viridis', vmin=0, vmax=1, shading='auto')
    ax.set_aspect('equal')
    ax.set_xlabel(r'$f_l$ (Hz)')
    ax.set_ylabel(r'$f_m$ (Hz)')
    ax.set_title(f'$|B_{{xxx}}(f_l, f_m)|$ — {title}')
    ax.axvline(f1, color='white', linewidth=0.3, alpha=0.4)
    ax.axhline(f2, color='white', linewidth=0.3, alpha=0.4)
fig.colorbar(im, ax=axes, label='magnitude normalizada')
plt.show()

l_target = int(round(f1 * N / fs))
m_target = int(round(f2 * N / fs))
print(f'|B(f1={f1:.0f}, f2={f2:.0f})| acoplado  = {B_coupled[l_target, m_target]:.3e}')
print(f'|B(f1={f1:.0f}, f2={f2:.0f})| não-acoplado = {B_uncoupled[l_target, m_target]:.3e}')
print(f'Razão acoplado / não-acoplado = '
      f'{B_coupled[l_target, m_target] / max(B_uncoupled[l_target, m_target], 1e-12):.1f}')

O contraste é o resultado central do tutorial. A população acoplada apresenta um pico nítido em $(f_l, f_m) = (50, 120)$ Hz — exatamente o ponto onde o produto $X(50)\,X(120)\,X^*(170)$ tem fase travada e a esperança não se anula. A população não acoplada mostra magnitude essencialmente residual na mesma coordenada, dominada apenas pela variância finita do estimador (proporcional a $1/\sqrt{K}$). A razão de magnitudes entre as duas populações no ponto-alvo é da ordem de uma a duas décadas, e cresce com $K$.

É essa assimetria que Sinha (2007, §4) explora para distinguir trinca de desalinhamento: não as alturas dos picos de 1X, 2X, 3X — idênticas na PSD entre as duas falhas —, mas *quais triadas $(f_l, f_m)$ exibem $|B|$ não-residual*.

## 4. Demonstração tipo-Sinha — $B_{22}$ contra $B_{13}$

Construímos duas populações de sinais que mimetizam, em escala simplificada, as assinaturas diagnósticas reportadas em Sinha (2007, Figs. 5–6):

- **Tipo-trinca:** três tons em 1X, 2X e 4X, com $\varphi(2\text{X}) = 2\varphi(1\text{X})$ e $\varphi(4\text{X}) = 2\varphi(2\text{X})$. Esta cadeia produz acoplamento em $B_{11}$ (porque $1\text{X} + 1\text{X} = 2\text{X}$) e em $B_{22}$ (porque $2\text{X} + 2\text{X} = 4\text{X}$).
- **Tipo-desalinhamento:** quatro tons em 1X, 2X, 3X, 4X, com $\varphi(2\text{X}) = 2\varphi(1\text{X})$ (mantém $B_{11}$), mas $\varphi(3\text{X})$ independente e $\varphi(4\text{X}) = \varphi(1\text{X}) + \varphi(3\text{X})$. Esta configuração produz acoplamento em $B_{11}$ e em $B_{13}$ (porque $1\text{X} + 3\text{X} = 4\text{X}$), e **ausência** de $B_{22}$ — pois $\varphi(4\text{X}) \neq 2\varphi(2\text{X})$ a menos por coincidência.

A demonstração não pretende reproduzir a física de Mayes-Davies ou de acoplamentos de Xia et al.; ela mostra apenas que a *combinatória de fases* descrita em Sinha (2007, §4) é detectável em sinais sintéticos pelo aparato de HOS sem invocar nenhum modelo mecânico.

In [ ]:
f_1x = 30.0
f_2x = 2.0 * f_1x
f_3x = 3.0 * f_1x
f_4x = 4.0 * f_1x
amp = 1.0


def make_crack_like(K: int, rng: np.random.Generator) -> np.ndarray:
    ens = np.empty((K, N))
    for k in range(K):
        p1 = rng.uniform(0.0, 2.0 * np.pi)
        p2 = 2.0 * p1                                 # phi(2X) = 2 phi(1X)  -> B11
        p4 = 2.0 * p2                                 # phi(4X) = 2 phi(2X)  -> B22
        s = (amp * np.cos(2.0 * np.pi * f_1x * t + p1)
             + amp * np.cos(2.0 * np.pi * f_2x * t + p2)
             + amp * np.cos(2.0 * np.pi * f_4x * t + p4)
             + noise_std * rng.standard_normal(N))
        ens[k] = s
    return ens


def make_misalignment_like(K: int, rng: np.random.Generator) -> np.ndarray:
    ens = np.empty((K, N))
    for k in range(K):
        p1 = rng.uniform(0.0, 2.0 * np.pi)
        p2 = 2.0 * p1                                 # phi(2X) = 2 phi(1X)  -> B11
        p3 = rng.uniform(0.0, 2.0 * np.pi)            # phi(3X) independente
        p4 = p1 + p3                                  # phi(4X) = phi(1X) + phi(3X) -> B13
        s = (amp * np.cos(2.0 * np.pi * f_1x * t + p1)
             + amp * np.cos(2.0 * np.pi * f_2x * t + p2)
             + amp * np.cos(2.0 * np.pi * f_3x * t + p3)
             + amp * np.cos(2.0 * np.pi * f_4x * t + p4)
             + noise_std * rng.standard_normal(N))
        ens[k] = s
    return ens


crack_like = make_crack_like(K, rng)
mis_like = make_misalignment_like(K, rng)

f_max_sinha = 180.0
freqs_s, B_crack = bispectrum_estimate(crack_like, fs, f_max_sinha)
_, B_mis = bispectrum_estimate(mis_like, fs, f_max_sinha)

Bmax_s = max(B_crack.max(), B_mis.max())
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
for ax, B, title in zip(axes, (B_crack, B_mis), ('tipo-trinca (espera-se $B_{11}$, $B_{22}$)',
                                                  'tipo-desalinhamento (espera-se $B_{11}$, $B_{13}$)')):
    im = ax.pcolormesh(freqs_s, freqs_s, B / Bmax_s, cmap='magma', vmin=0, vmax=1, shading='auto')
    ax.set_aspect('equal')
    ax.set_xlabel(r'$f_l$ (Hz)')
    ax.set_ylabel(r'$f_m$ (Hz)')
    ax.set_title(title)
    for f in (f_1x, f_2x, f_3x, f_4x):
        ax.axvline(f, color='cyan', linewidth=0.3, alpha=0.4)
        ax.axhline(f, color='cyan', linewidth=0.3, alpha=0.4)
fig.colorbar(im, ax=axes, label='magnitude normalizada')
plt.show()


def bin_of(f: float) -> int:
    return int(round(f * N / fs))


i_1x, i_2x, i_3x = bin_of(f_1x), bin_of(f_2x), bin_of(f_3x)
print(f'Tipo-trinca           : |B11| (l=m={f_1x:.0f}) = {B_crack[i_1x, i_1x]:.3e}   '
      f'|B22| (l=m={f_2x:.0f}) = {B_crack[i_2x, i_2x]:.3e}   '
      f'|B13| (l={f_1x:.0f}, m={f_3x:.0f}) = {B_crack[i_1x, i_3x]:.3e}')
print(f'Tipo-desalinhamento   : |B11| (l=m={f_1x:.0f}) = {B_mis[i_1x, i_1x]:.3e}   '
      f'|B22| (l=m={f_2x:.0f}) = {B_mis[i_2x, i_2x]:.3e}   '
      f'|B13| (l={f_1x:.0f}, m={f_3x:.0f}) = {B_mis[i_1x, i_3x]:.3e}')

Os mapas de biespectro reproduzem qualitativamente a assimetria diagnóstica reportada em Sinha (2007, Figs. 5–6):

- O sinal tipo-trinca exibe magnitude elevada na diagonal — picos em $B_{11}$ e $B_{22}$.
- O sinal tipo-desalinhamento exibe magnitude elevada em $B_{11}$ e em $B_{13}$ ($=B_{31}$), com $B_{22}$ no nível de ruído.

A PSD desses dois sinais — não mostrada para economia de espaço, mas verificável trivialmente — exibe picos em 1X, 2X, 3X e 4X em ambos os casos, com diferenças de altura apenas marginais (o tipo-trinca não tem 3X; o tipo-desalinhamento sim). O *padrão de acoplamento de fase*, e não o conteúdo de magnitude, é o que distingue categoricamente as duas falhas.

## Notas técnicas e correspondências com Sinha (2007)

- A promediação foi feita sobre $K$ realizações independentes em vez de sobre segmentos sobrepostos do mesmo sinal. Sob estacionariedade e ergodicidade, os dois procedimentos convergem para o mesmo estimador. Sinha (2007, p. 329) usa o segundo: 50 segmentos com 50% de sobreposição, $\Delta f = 1{,}25$ Hz, sobre um único registro experimental.
- Nenhum janelamento (Hann, Hamming, etc.) foi aplicado, pois cada realização é um trecho com início e fim em fase aleatória, e o sinal tem componentes em frequências exatamente comensuráveis com $f_s/N$, eliminando vazamento espectral. Em sinais reais, o janelamento é obrigatório e introduz suas próprias considerações — a propriedade de multiplicação no tempo $=$ convolução em frequência (Oppenheim, Willsky e Nawab, 1997, §5.5, p. 388) governa o trade-off entre resolução e vazamento.
- O triespectro, definido em Sinha (2007, Eq. 3) sobre triadas $(f_l, f_m, f_n)$, não é demonstrado aqui pois requer visualização 3D que excede o escopo didático deste caderno. Sua generalização operacional segue a mesma lógica: produto quádruplo, esperança, condição $l + m + n \leq N$, e magnitude não-residual onde houver acoplamento cúbico de fase.

## Referências

Oppenheim, A. V., Willsky, A. S., & Nawab, S. H. (1997). *Signals and systems* (2nd ed.). Prentice Hall.

Sinha, J. K. (2007). Higher order spectra for crack and misalignment identification in the shaft of a rotating machine. *Structural Health Monitoring*, *6*(4), 325–334. https://doi.org/10.1177/1475921707082309